# 02 — Silver: Yellow Taxi

Reads Yellow Taxi records from Bronze (tlc_bronze.yellow_raw), applies data quality rules, enriches with zone metadata via broadcast join, builds the normalized Silver schema, and writes valid records to tlc_silver.trips_yellow. Rejected records are routed to tlc_audit.quarantine.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import YELLOW_GREEN_RULES, apply_rules, print_rejection_summary
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_yellow_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F

YEARS_TO_PROCESS = [2024, 2025]
print("Imports OK.")

In [ ]:
spark = get_spark(app_name="TLC_Silver_Yellow")

## Start Audit Control

In [ ]:
control = ControlManager("silver_yellow")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "yellow"})
run_id = execution.execution_id
print(f"Execution ID: {run_id}")

## Read from Bronze

In [ ]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "yellow_raw")

# Filter to requested years if year column exists
if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("tpep_pickup_datetime").isin(YEARS_TO_PROCESS))

records_in = df_bronze.count()
print(f"Records read from Bronze: {records_in:,}")

## Apply Data Quality Rules

In [ ]:
valid_df, rejected_df = apply_rules(df_bronze, YELLOW_GREEN_RULES)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="yellow_quality_rules",
    dataset=f"yellow_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

## Route Rejected Records to Quarantine

In [ ]:
if records_rejected > 0:
    audit_db = get_audit_db()
    quarantine_col = audit_db["quarantine"]

    quarantine_docs = [
        {
            "quarantined_at": __import__('datetime').datetime.utcnow(),
            "run_id": run_id,
            "pipeline": "silver_yellow",
            "reason": row["_rejection_reason"],
            "raw_record": {k: v for k, v in row.asDict().items() if k != "_rejection_reason"},
        }
        for row in rejected_df.limit(10000).collect()  # cap at 10k for large batches
    ]
    if quarantine_docs:
        quarantine_col.insert_many(quarantine_docs)
        print(f"Quarantined {len(quarantine_docs):,} records into tlc_audit.quarantine")
else:
    print("No records quarantined.")

## Enrich with Zone Metadata

In [ ]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

## Build Silver Schema & Write to MongoDB

In [ ]:
silver_df = build_yellow_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_yellow", mode="append")
print(f"Rows written to tlc_silver.trips_yellow: {n_written:,}")

In [ ]:
control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
    },
)

## Audit Report

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))